# 📖 Notebook 1: Metrics Collection & Time-Series Storage

Before we build dashboards or alerts, we need to understand **what metrics are**, **how they're collected**, and **how time-series databases store them**.

## Learning Objectives

By the end of this notebook, you'll understand:
- What a metric is (name, labels, value, timestamp)
- The difference between counters, gauges, and histograms
- How Prometheus scrapes metrics from HTTP endpoints (pull model)
- How time-series data is stored and why specialized databases exist
- How to query metrics using PromQL

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/metrics-monitoring
docker compose up -d
```

### Visualization Tools

- **Prometheus**: http://localhost:9090 — Run PromQL queries, check scrape targets
- **Grafana**: http://localhost:3000 — Login: admin / admin
- **Adminer** (PostgreSQL): http://localhost:8080 — Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `metrics_demo`
- **RedisInsight** (Redis): http://localhost:5540 — Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import requests
import time
import json
import threading
import random
import math
from http.server import HTTPServer, BaseHTTPRequestHandler

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "metrics_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

PROMETHEUS_URL = "http://localhost:9090"

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

try:
    resp = requests.get(f"{PROMETHEUS_URL}/-/healthy", timeout=5)
    print("✅ Connected to Prometheus")
except Exception as e:
    print(f"❌ Prometheus failed: {e}")
    print("   Run: docker compose up -d")

## 🤔 What Is a Metric?

A **metric** is a numerical measurement taken at a point in time. Every metric has four parts:

```
cpu_usage{host="server-1", region="us-east"} = 0.75   @ 1640000000
  ↑ name        ↑ labels                       ↑ value   ↑ timestamp
```

- **Name**: What you're measuring (e.g., `cpu_usage`, `http_requests_total`)
- **Labels**: Key-value pairs that identify WHERE the metric came from (e.g., `host="server-1"`)
- **Value**: The actual number (e.g., `0.75` = 75% CPU)
- **Timestamp**: When it was measured (Unix epoch seconds)

### What Is a Series?

A **series** is one unique combination of metric name + labels, tracked over time:

```
cpu_usage{host="server-1"} = [0.75, 0.80, 0.72, 0.85, ...]
cpu_usage{host="server-2"} = [0.60, 0.55, 0.62, 0.58, ...]
```

If you have 500,000 servers each reporting `cpu_usage`, that's **500,000 separate series**.  
This is why **cardinality** (the number of unique series) is the central scaling challenge.

In [ ]:
# Let's see what a metric data point looks like in Python

metric_data_point = {
    "name": "cpu_usage",
    "labels": {"host": "server-1", "region": "us-east"},
    "value": 0.75,
    "timestamp": int(time.time())
}

print("📊 A single metric data point:")
print(json.dumps(metric_data_point, indent=2))
print()

# At scale, each data point is tiny (~100-200 bytes)
# But at 5 million per second, that's ~1 GB/s of raw ingestion!
point_size = len(json.dumps(metric_data_point).encode())
print(f"📏 Size of one data point: {point_size} bytes")
print(f"   At 5M points/sec: {point_size * 5_000_000 / 1_000_000_000:.1f} GB/sec")
print()

# Series = unique combo of name + labels
# Each host creates its own series
hosts = ["server-1", "server-2", "server-3"]
metrics = ["cpu_usage", "memory_usage", "disk_usage"]
series_count = len(hosts) * len(metrics)
print(f"🔢 {len(hosts)} hosts × {len(metrics)} metrics = {series_count} unique series")
print(f"   500,000 hosts × 100 metrics = {500_000 * 100:,} series (production scale!)")

## 📊 Metric Types

Not all metrics work the same way. There are three main types:

| Type | Description | Example | Key Property |
|------|-------------|---------|-------------|
| **Gauge** | A value that goes up and down | CPU usage, memory %, temperature | Can be any number |
| **Counter** | A value that only goes up | Total requests, total errors, bytes sent | Monotonically increasing |
| **Histogram** | Distribution of values in buckets | Request latency, response sizes | Shows percentiles (p50, p99) |

Understanding the type matters because it determines **how you query**:
- Gauge → use `avg()`, `max()`, `min()` directly
- Counter → use `rate()` to get per-second change (raw value always goes up)
- Histogram → use `histogram_quantile()` for percentiles

In [ ]:
# Let's simulate all three metric types.
# Seeded so the numbers below (and the assertions at the end) are reproducible.
random.seed(7)

print("📊 Gauge: CPU Usage (goes up and down)")
print("=" * 50)
gauge_values = []
cpu = 50.0
for i in range(10):
    cpu += random.uniform(-10, 10)  # fluctuates randomly
    cpu = max(0, min(100, cpu))     # clamp between 0-100
    gauge_values.append(round(cpu, 1))
print(f"  Values: {gauge_values}")
print(f"  Notice: values go UP and DOWN — that's a gauge!")
print()

print("📊 Counter: Total HTTP Requests (only goes up)")
print("=" * 50)
counter_values = []
total = 0
for i in range(10):
    total += random.randint(50, 200)  # new requests each interval
    counter_values.append(total)
print(f"  Values: {counter_values}")
print(f"  Notice: values ONLY GO UP — that's a counter!")

# rate() is (last sample - first sample) / (time between them). Careful: 10
# samples span 9 gaps, not 10 — a classic off-by-one when you compute it by hand.
INTERVAL_S = 10
delta = counter_values[-1] - counter_values[0]
gaps = len(counter_values) - 1
print(f"  Increase across the window: {delta} requests over {gaps} intervals ({gaps * INTERVAL_S}s)")
print(f"  rate() = {delta / (gaps * INTERVAL_S):.1f} requests/sec")
print()

print("📊 Histogram: Request Latency Distribution")
print("=" * 50)
latencies = [random.expovariate(1/0.5) for _ in range(100)]  # most fast, some slow
latencies.sort()
p50 = latencies[49]
p90 = latencies[89]
p99 = latencies[98]
print(f"  p50 (median): {p50:.3f}s — half of requests are faster than this")
print(f"  p90:          {p90:.3f}s — 90% of requests are faster")
print(f"  p99:          {p99:.3f}s — 99% of requests are faster")
print(f"  💡 p99 is {p99 / p50:.1f}× the median here — this is why we monitor percentiles!")

# Guard rails: if any of these ever trip, the demo above stopped demonstrating
# what the prose says it demonstrates.
assert counter_values == sorted(counter_values), "a counter must never decrease"
assert min(gauge_values) < max(gauge_values), "a gauge demo must actually move up and down"
assert p50 < p90 < p99, "percentiles must be monotonic"
assert p99 / p50 > 3, f"expected a heavy tail, got p99/p50 = {p99 / p50:.1f}"


## ⚠️ Two Ways to Get Percentiles Wrong

Percentiles are the most useful numbers on a latency dashboard and the easiest to compute
incorrectly. Two mistakes account for almost every wrong p99 you will ever see.

### Mistake 1: Averaging percentiles

`avg(p99_by_instance)` is **not** the p99 of the fleet. A percentile is a rank statistic —
you cannot recover the rank of a merged population from the ranks of its parts. The only
correct way is to merge the underlying observations (or the histogram buckets) *first*,
then take the quantile.

> **Rule**: you may sum counters and average gauges across instances. You may never
> average percentiles. Aggregate the buckets, then compute the quantile.

### Mistake 2: Trusting `histogram_quantile()` beyond your bucket resolution

Prometheus histograms do not store observations. They store **cumulative counters per
bucket**, and `histogram_quantile()` linearly interpolates *inside* whichever bucket the
target rank lands in. That has two consequences:

- Your answer is only as precise as the bucket boundaries around it. A `(2.5, 5.0]` bucket
  gives you a p99 that could be off by seconds.
- If the rank lands in the `+Inf` bucket — i.e. your highest **finite** bucket is below the
  real p99 — Prometheus returns the highest finite bound. The number looks plausible and is
  completely meaningless.

Let's measure both effects on data we control, so we know the true answer to compare against.


In [ ]:
# Three instances behind one load balancer. web-3 is sick and also takes less
# traffic — the classic shape that makes averaged percentiles lie.
random.seed(1337)

def nearest_rank(sorted_vals: list, q: float) -> float:
    """Exact quantile from raw observations (nearest-rank definition)."""
    return sorted_vals[max(0, math.ceil(q * len(sorted_vals)) - 1)]

instances = {
    "web-1": [random.expovariate(1 / 0.20) for _ in range(4000)],
    "web-2": [random.expovariate(1 / 0.20) for _ in range(4000)],
    "web-3": [random.expovariate(1 / 0.20) + random.uniform(0, 3.0) for _ in range(2000)],
}

per_host_p99 = {h: nearest_rank(sorted(v), 0.99) for h, v in instances.items()}
pooled = sorted(v for vals in instances.values() for v in vals)
true_p99 = nearest_rank(pooled, 0.99)                       # merge first, THEN rank
avg_of_p99 = sum(per_host_p99.values()) / len(per_host_p99)  # the wrong way

print("❌ Mistake 1: averaging percentiles across instances")
print("=" * 62)
for h, v in per_host_p99.items():
    print(f"  p99({h}) = {v:.3f}s   ({len(instances[h]):,} requests)")
print(f"  avg of the three p99s : {avg_of_p99:.3f}s   <- what a naive dashboard shows")
print(f"  TRUE fleet-wide p99   : {true_p99:.3f}s   <- what your users actually feel")
print(f"  Error: {100 * (avg_of_p99 - true_p99) / true_p99:+.0f}% — you would under-report the tail")
print()


def histogram_quantile(q: float, bounds: list, cumulative: list, total: int) -> float:
    """
    Reimplementation of Prometheus `histogram_quantile()` for classic histograms.
    `cumulative[i]` = number of observations <= bounds[i]. Anything above
    bounds[-1] lives in the implicit +Inf bucket.
    """
    if total == 0:
        return float("nan")
    rank = q * total
    if rank > cumulative[-1]:
        # The quantile falls in the +Inf bucket. Prometheus has no upper edge to
        # interpolate towards, so it returns the highest FINITE bound. This is the
        # silent-garbage case.
        return bounds[-1]
    i = 0
    while cumulative[i] < rank:
        i += 1
    lower, lower_count = (0.0, 0) if i == 0 else (bounds[i - 1], cumulative[i - 1])
    upper, upper_count = bounds[i], cumulative[i]
    if upper_count == lower_count:
        return upper
    # Linear interpolation INSIDE the bucket — this is where the error comes from.
    return lower + (upper - lower) * (rank - lower_count) / (upper_count - lower_count)


def bucketize(vals: list, bounds: list) -> list:
    """Cumulative bucket counts, exactly what a Prometheus `_bucket` series holds."""
    return [sum(1 for v in vals if v <= b) for b in bounds]


layouts = {
    "client-library defaults": [.005, .01, .025, .05, .075, .1, .25, .5, .75, 1.0, 2.5, 5.0, 7.5, 10.0],
    "highest bucket = 1s":     [.005, .01, .025, .05, .1, .25, .5, 1.0],
    "tuned around the SLO":    [.05, .1, .25, .5, 1.0, 1.5, 2.0, 2.5, 2.75, 3.0, 3.25, 3.5, 4.0, 5.0, 10.0],
}

print("❌ Mistake 2: reading a p99 your buckets cannot resolve")
print("=" * 62)
print(f"  TRUE p99 (from raw observations): {true_p99:.3f}s")
print()
print(f"  {'Bucket layout':<26}{'top finite':>11}{'p99 estimate':>14}{'error':>9}")
print("  " + "-" * 58)
estimates = {}
for name, bounds in layouts.items():
    est = histogram_quantile(0.99, bounds, bucketize(pooled, bounds), len(pooled))
    estimates[name] = est
    err = 100 * (est - true_p99) / true_p99
    print(f"  {name:<26}{bounds[-1]:>10.1f}s{est:>13.3f}s{err:>8.0f}%")

print()
print("💡 The default buckets jump straight from 2.5s to 5.0s, so the p99 is")
print("   interpolated across a 2.5-second-wide gap — it lands well off the truth.")
print("   The 1s layout is worse: the answer is simply the top bucket edge, and the")
print("   graph will sit pinned at exactly 1.000s no matter how bad latency gets.")
print("   Pick buckets around the SLO you care about, and always keep a finite")
print("   bucket above your worst realistic latency.")

# Guard rails — these encode the lesson, not the implementation.
assert avg_of_p99 < true_p99 * 0.75, (
    f"averaging percentiles should badly under-report the tail; "
    f"got avg={avg_of_p99:.3f} vs true={true_p99:.3f}")
assert estimates["highest bucket = 1s"] == layouts["highest bucket = 1s"][-1], (
    "when the quantile falls in +Inf, histogram_quantile must return the top finite bound")
tuned_err = abs(estimates["tuned around the SLO"] - true_p99) / true_p99
default_err = abs(estimates["client-library defaults"] - true_p99) / true_p99
assert tuned_err < 0.05, f"tuned buckets should land within 5% of truth, got {tuned_err:.1%}"
assert default_err > 4 * tuned_err, (
    f"coarse buckets should be clearly worse than tuned ones; "
    f"default={default_err:.1%} tuned={tuned_err:.1%}")


## 💥 Cardinality: The #1 Scaling Problem

**Cardinality** = the number of **unique series** (unique combinations of metric name + labels).

Every unique label value creates a **new series**. Time-series databases store one chunk of memory per active series, so cardinality directly determines RAM, disk, and query cost.

### The Trap: Using High-Cardinality Values as Labels

| Label choice | Cardinality | Verdict |
|---|---|---|
| `host="web-1"` (500 hosts) | 500 series | ✅ Safe |
| `status="500"` (5 codes) | 5 series | ✅ Safe |
| `user_id="u_42"` (10M users) | 10M series | ❌ Explodes |
| `request_id="uuid..."` (per request) | Unbounded | 💀 Kills the DB |

### Rule of Thumb

> **Labels identify WHERE a metric came from, not WHAT happened.**
> If a label's possible values are unbounded (user IDs, URLs with query strings, error messages),
> it does not belong on a metric — it belongs in logs or traces.


In [ ]:
# Demonstrate cardinality: 1 metric can become millions of series

scenarios = [
    ("Good:   metric by host+region",
     {"host": 500, "region": 5}),
    ("Good:   HTTP metric by status+endpoint",
     {"status": 5, "endpoint": 20}),
    ("Bad:    add user_id label",
     {"host": 500, "region": 5, "user_id": 10_000_000}),
    ("Worst:  add request_id label (unbounded)",
     {"host": 500, "region": 5, "request_id": 1_000_000_000}),
]

header = f"{'Scenario':<45} {'Unique series':>18}  Memory (~3KB/series)"
print(header)
print("-" * len(header))
for name, labels in scenarios:
    series = 1
    for v in labels.values():
        series *= v
    mem_gb = series * 3_000 / 1e9
    warn = "  [WARN] explodes the TSDB" if series > 1_000_000 else ""
    print(f"{name:<45} {series:>18,}  {mem_gb:>8.2f} GB{warn}")

print()
print("Production Prometheus typically runs fine up to ~1-5M active series per node.")
print("A single bad label like user_id can push you past that instantly.")
print("Fix: drop the label and record it in logs/traces, or bucket it")
print("(e.g., 'user_tier=premium' has only 3 values).")


## 🔄 How Prometheus Collects Metrics (Pull Model)

Prometheus uses a **pull model**: it scrapes (fetches) metrics from HTTP endpoints at regular intervals.

```
┌──────────┐    GET /metrics     ┌─────────────┐
│Prometheus │ ──────────────────► │ Your App    │
│ (scraper) │ ◄────────────────── │ (port 8000) │
│           │   text/plain        │             │
└──────────┘   metric lines       └─────────────┘
     │
     │ stores
     ▼
┌──────────┐
│ TSDB     │  (Time-Series Database built into Prometheus)
└──────────┘
```

**How it works:**
1. Your app exposes a `/metrics` HTTP endpoint
2. Prometheus fetches that endpoint every N seconds (configured in `prometheus.yml`)
3. The response is plain text in a specific format
4. Prometheus parses the text and stores each metric as a time-series

**Why pull and not push?**
- Prometheus controls the pace (no overwhelming the server)
- Easy to tell if a target is down (scrape fails)
- No need for the app to know where to send data

Let's build a `/metrics` endpoint and watch Prometheus scrape it!

In [ ]:
# Build a simple metrics endpoint that Prometheus can scrape.
# This simulates a server reporting CPU, memory, latency, etc.

from prometheus_client import (
    Gauge, Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST,
    CollectorRegistry
)

# Create a fresh registry (avoids conflicts if you re-run this cell)
registry = CollectorRegistry()

# Define our metrics — these are the "instruments" that produce data
cpu_gauge = Gauge('demo_cpu_usage', 'CPU usage percentage',
                  ['host', 'region'], registry=registry)
memory_gauge = Gauge('demo_memory_usage', 'Memory usage percentage',
                     ['host', 'region'], registry=registry)
request_latency = Gauge('demo_request_latency', 'Average request latency in seconds',
                        ['host'], registry=registry)
error_rate = Gauge('demo_error_rate', 'Error rate percentage',
                   ['host'], registry=registry)

# A real Histogram, so we can compute percentiles instead of guessing from the
# average. Buckets are explicit and chosen to bracket this workload's tail —
# the client-library defaults jump 2.5s -> 5.0s, which (as we measured above)
# is far too coarse to resolve a p99 that lives in that range.
request_duration = Histogram(
    'demo_request_duration_seconds', 'HTTP request duration in seconds',
    ['host'],
    buckets=(0.05, 0.1, 0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 7.5, 10.0),
    registry=registry)
http_requests = Counter('demo_http_requests_total', 'Total HTTP requests',
                        ['host', 'status'], registry=registry)

# Simulated servers
SERVERS = [
    {"host": "web-1", "region": "us-east"},
    {"host": "web-2", "region": "us-east"},
    {"host": "web-3", "region": "us-west"},
    {"host": "api-1", "region": "us-east"},
    {"host": "api-2", "region": "eu-west"},
]

# Simulate realistic metric values that change over time
sim_tick = 0

def update_metrics():
    """Generate realistic-looking metric values."""
    global sim_tick
    sim_tick += 1
    for server in SERVERS:
        h, r = server["host"], server["region"]
        # CPU: sine wave + noise (simulates daily load patterns).
        # Amplitude is deliberately tuned so the crest sits ABOVE 80% for ~95
        # seconds of every ~10.5-minute cycle. That matters: `alert_rules.yml`
        # has a `HighCpuUsage` rule with `for: 1m`, and a wave that peaked at
        # 75% would leave that rule permanently inactive — an alerting lab
        # whose alerts never fire teaches nothing.
        base_cpu = 55 + 35 * math.sin(sim_tick / 20)
        cpu_val = base_cpu + random.uniform(-6, 6)
        cpu_gauge.labels(host=h, region=r).set(max(0, min(100, cpu_val)))

        # Memory: a sawtooth — climbs steadily, then drops back when the process
        # restarts. (Note it RESETS: this is a restart-on-OOM pattern, not an
        # unbounded leak.) Tuned to spend ~70s per cycle above the 85% threshold
        # so `HighMemoryUsage` (`for: 1m`) is reachable too.
        mem_val = 45 + (sim_tick * 0.35) % 50 + random.uniform(-5, 5)
        memory_gauge.labels(host=h, region=r).set(max(0, min(100, mem_val)))

        # Latency: draw a batch of individual request durations — most fast, ~5%
        # on a slow path. We record the batch BOTH ways: the gauge gets the mean
        # (matching its 'Average request latency' help text) and the histogram
        # gets every observation. Notebook 3 compares what the two tell you.
        batch = []
        for _ in range(20):
            d = 0.1 + random.expovariate(5)
            if random.random() < 0.05:  # 5% chance of a slow request
                d += random.uniform(1, 5)
            request_duration.labels(host=h).observe(d)
            batch.append(d)
        request_latency.labels(host=h).set(round(sum(batch) / len(batch), 3))

        # Error rate: mostly low, sometimes spikes
        err = random.expovariate(2)
        if random.random() < 0.03:  # 3% chance of error spike
            err += random.uniform(5, 15)
        error_rate.labels(host=h).set(round(max(0, err), 2))

        # HTTP requests: always incrementing (counter)
        http_requests.labels(host=h, status="200").inc(random.randint(50, 200))
        http_requests.labels(host=h, status="500").inc(random.randint(0, 5))

print("✅ Metric definitions created")
print(f"   Simulating {len(SERVERS)} servers: {[s['host'] for s in SERVERS]}")
print("   Gauges: cpu, memory, avg latency, error rate | Counter: http_requests")
print("   Histogram: demo_request_duration_seconds (15 explicit buckets)")


In [ ]:
# Start an HTTP server that serves /metrics for Prometheus to scrape.
# This runs in the background so you can keep using the notebook.

class MetricsHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/metrics":
            update_metrics()  # generate fresh values on each scrape
            output = generate_latest(registry)
            self.send_response(200)
            self.send_header("Content-Type", CONTENT_TYPE_LATEST)
            self.end_headers()
            self.wfile.write(output)
        else:
            self.send_response(404)
            self.end_headers()

    def log_message(self, format, *args):
        pass  # suppress request logs

# Start server in background thread (idempotent — safe to re-run this cell)
_prev = globals().get("server")
if _prev is not None:
    try:
        _prev.shutdown(); _prev.server_close()
        print("♻️  Previous metrics server stopped — restarting.")
    except Exception:
        pass

server = HTTPServer(("0.0.0.0", 8000), MetricsHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()

print("✅ Metrics server running on http://localhost:8000/metrics")
print("   Prometheus is configured to scrape this every 5 seconds.")
print()
print("👀 Try it yourself:")
print("   1. Open http://localhost:8000/metrics in your browser")
print("   2. Open Prometheus: http://localhost:9090/targets")
print("      You should see 'demo_app' target with state UP")

In [ ]:
# Let's see what the /metrics endpoint looks like — this is exactly
# what Prometheus sees when it scrapes our app.

resp = requests.get("http://localhost:8000/metrics")
lines = resp.text.strip().split("\n")

print("📄 Raw /metrics output (first 25 lines):")
print("=" * 70)
for line in lines[:25]:
    print(line)
print("...")
print()
print(f"Total lines: {len(lines)}")
print()
print("💡 This is plain text! Each line is one metric with labels and a value.")
print("   Lines starting with # are HELP (description) and TYPE (gauge/counter/etc).")

## 💾 Time-Series Storage: Why Not Just Use Postgres?

Let's store the same metrics in both Postgres and see why specialized time-series databases exist.

### The Naive Approach: Postgres

```sql
CREATE TABLE metrics (
    id SERIAL,
    metric_name TEXT,
    labels JSONB,
    value DOUBLE PRECISION,
    timestamp TIMESTAMPTZ
);
```

This works fine for small scale. But at 5M writes/sec:
- Postgres can't sustain that write throughput
- Indexes bloat as data grows
- DELETEs for retention cause vacuum pressure
- Queries over weeks of data are painfully slow

### Time-Series Databases Are Optimized For This

| Feature | Postgres | Time-Series DB (Prometheus/InfluxDB) |
|---------|----------|-------------------------------------|
| Write pattern | Random inserts | Append-only (sequential) |
| Compression | General purpose | 10-20× (timestamps + values compress well) |
| Retention | Manual DELETE + VACUUM | Drop old time chunks instantly |
| Query speed | Scans entire table | Skips irrelevant time ranges |
| Rollups | Manual materialized views | Varies — see note below |

> ⚠️ **Careful with that last row.** Vanilla Prometheus has *no* built-in downsampling at
> all: samples are stored at scrape resolution until they age out of the retention window
> (this lab runs `--storage.tsdb.retention.time=15d`). Rollups are exactly what Thanos,
> Cortex and Grafana Mimir bolt on top of it; InfluxDB and TimescaleDB do have their own
> continuous-aggregate mechanisms. "Time-series DB" is not one product with one feature set.


In [ ]:
# Let's compare: write metrics to Postgres and measure performance

conn = get_db_connection()
cursor = conn.cursor()

# Create a simple metrics table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS metrics_raw (
        id SERIAL PRIMARY KEY,
        metric_name TEXT NOT NULL,
        labels JSONB,
        value DOUBLE PRECISION NOT NULL,
        ts TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_metrics_ts ON metrics_raw(ts)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_metrics_name ON metrics_raw(metric_name)")
conn.commit()

random.seed(11)
rows = [("cpu_usage", json.dumps({"host": f"server-{i % 100}"}), random.uniform(0, 100))
        for i in range(10_000)]

# Attempt 1: the naive way — one round trip per data point.
start = time.time()
for row in rows:
    cursor.execute(
        "INSERT INTO metrics_raw (metric_name, labels, value, ts) VALUES (%s, %s, %s, NOW())",
        row
    )
conn.commit()
naive_time = time.time() - start
naive_rate = len(rows) / naive_time

# Attempt 2: the same 10,000 rows batched into multi-row INSERTs. This is the
# fair comparison — quoting the row-at-a-time number alone would be measuring
# network round trips, not Postgres.
from psycopg2.extras import execute_values

start = time.time()
execute_values(
    cursor,
    "INSERT INTO metrics_raw (metric_name, labels, value, ts) VALUES %s",
    rows,
    template="(%s, %s, %s, NOW())",
    page_size=1000,
)
conn.commit()
batch_time = time.time() - start
batch_rate = len(rows) / batch_time

print(f"📝 Row-at-a-time INSERT: {naive_time:.2f}s -> {naive_rate:,.0f} writes/sec")
print(f"📝 Batched INSERT:       {batch_time:.2f}s -> {batch_rate:,.0f} writes/sec")
print(f"   Batching alone bought us {batch_rate / naive_rate:.0f}×.")
print()
print(f"   Target for a real metrics platform: 5,000,000 writes/sec.")
print(f"   Even batched, one Postgres node is ~{5_000_000 / batch_rate:,.0f}× short.")
print()
print("💡 Be honest about what this measures. Postgres is not slow — a tuned node")
print("   doing COPY can hit six figures per second. The reasons metrics platforms")
print("   don't use it are structural, not raw insert speed:")
print("   • every row carries the full label set again (no per-series dictionary)")
print("   • B-tree indexes must be kept sorted as data arrives")
print("   • retention means DELETE + VACUUM instead of dropping a whole time chunk")
print("   • no columnar delta-of-delta / XOR compression for timestamps and values")

assert batch_rate > naive_rate, (
    f"batching should beat row-at-a-time inserts; got {batch_rate:,.0f} vs {naive_rate:,.0f}/sec")


In [ ]:
# Now let's query Postgres and see how it handles time-range queries

# Query 1: Average CPU for all servers in the last minute
start = time.time()
cursor.execute("""
    SELECT 
        labels->>'host' AS host,
        AVG(value) AS avg_cpu,
        COUNT(*) AS samples
    FROM metrics_raw
    WHERE metric_name = 'cpu_usage'
      AND ts >= NOW() - INTERVAL '1 minute'
    GROUP BY labels->>'host'
    ORDER BY avg_cpu DESC
    LIMIT 10
""")
results = cursor.fetchall()
query_time = (time.time() - start) * 1000

print(f"📊 Top 10 hosts by CPU (last 1 minute) — {query_time:.1f}ms")
print(f"{'Host':<15} {'Avg CPU':>10} {'Samples':>10}")
print("-" * 40)
for host, avg_cpu, samples in results:
    print(f"{host:<15} {avg_cpu:>9.1f}% {samples:>10}")

print()
print("💡 This works for 10K rows. But imagine 30 days of data at 5M/sec...")
print(f"   That's {5_000_000 * 86400 * 30:,.0f} rows ({5_000_000 * 86400 * 30 * 100 / 1e12:.1f} TB)")
print("   This query would take minutes, not milliseconds!")

conn.close()

## 🔍 Querying Prometheus with PromQL

Now let's query the **real** time-series database — Prometheus. It has its own query language called **PromQL**.

Wait about 30 seconds after starting the metrics server so Prometheus has time to collect some data.

### Common PromQL Patterns

| Query | What It Does |
|-------|--------------|
| `demo_cpu_usage` | Get the latest value for all series |
| `demo_cpu_usage{host="web-1"}` | Filter by label |
| `avg(demo_cpu_usage)` | Average across all hosts |
| `avg by (region)(demo_cpu_usage)` | Average grouped by region |
| `rate(demo_http_requests_total[1m])` | Requests per second (over 1 min window) |
| `max_over_time(demo_cpu_usage[5m])` | Peak CPU in last 5 minutes |
| `histogram_quantile(0.99, sum by (le)(rate(demo_request_duration_seconds_bucket[5m])))` | Fleet-wide p99 latency |

### Sizing a `rate()` window

`rate()` needs at least two samples inside the range to compute anything, and it drops the
range entirely if it finds fewer. The working rule is **make the range at least 4× the scrape
interval**. Our `demo_app` job scrapes every 5 seconds, so `[1m]` (12 samples) is comfortable;
`[10s]` (2 samples) would produce a jagged line full of gaps the moment a single scrape failed.

`rate()` also handles **counter resets** for you: if the process restarts and
`demo_http_requests_total` drops back to 0, `rate()` detects the decrease and treats it as a
reset rather than reporting a huge negative rate. That is exactly why you must never apply
`rate()` to a gauge — a gauge going down is normal, and `rate()` would silently "correct" it.


In [ ]:
# Helper function to query Prometheus

def prom_query(query: str) -> dict:
    """Run a PromQL instant query and return parsed results."""
    resp = requests.get(f"{PROMETHEUS_URL}/api/v1/query", params={"query": query})
    data = resp.json()
    if data["status"] != "success":
        print(f"❌ Query failed: {data.get('error', 'unknown error')}")
        return []
    return data["data"]["result"]

def prom_range_query(query: str, start: float, end: float, step: str = "15s") -> dict:
    """Run a PromQL range query and return parsed results."""
    resp = requests.get(f"{PROMETHEUS_URL}/api/v1/query_range", params={
        "query": query, "start": start, "end": end, "step": step
    })
    data = resp.json()
    if data["status"] != "success":
        print(f"❌ Query failed: {data.get('error', 'unknown error')}")
        return []
    return data["data"]["result"]

def wait_for_series(query: str, timeout: int = 90, poll: float = 2.0) -> list:
    """
    Block until a PromQL query returns something, or give up after `timeout`.

    A fixed `time.sleep(15)` is the wrong tool here. Prometheus scrapes our app
    every 5 seconds, but the first scrape lands somewhere in that window, the
    container may still be starting, and on a busy machine everything stretches.
    Sleeping a guessed constant means the rest of this notebook sometimes prints
    nothing at all and looks broken. Poll for the thing you actually need.
    """
    deadline = time.time() + timeout
    while time.time() < deadline:
        results = prom_query(query)
        if results:
            return results
        time.sleep(poll)
    return []


print("⏳ Waiting for Prometheus to scrape our /metrics endpoint...")
if wait_for_series("demo_cpu_usage"):
    print("   ✅ got data")
else:
    print("   ⚠️  still nothing after 90s. Check http://localhost:9090/targets —")
    print("      the 'demo_app' target should be UP. On Linux, host.docker.internal")
    print("      may not resolve; the compose file relies on it to reach port 8000.")

# Query 1: Get latest CPU for all hosts
print("\n📊 Query: demo_cpu_usage (latest values)")
print("=" * 60)
results = prom_query("demo_cpu_usage")
for r in results:
    labels = r["metric"]
    value = float(r["value"][1])
    print(f"  {labels.get('host', '?'):<10} region={labels.get('region', '?'):<10} CPU={value:.1f}%")

print()
print("💡 Each line is a separate SERIES (unique host + region combination)")


In [ ]:
# More PromQL queries — filtering, aggregation, and rates

# Query 2: Average CPU by region
print("📊 Query: avg by (region)(demo_cpu_usage)")
print("=" * 50)
results = prom_query('avg by (region)(demo_cpu_usage)')
for r in results:
    region = r["metric"].get("region", "unknown")
    value = float(r["value"][1])
    print(f"  region={region:<12} avg CPU={value:.1f}%")

print()

# Query 3: HTTP request rate per host
print("📊 Query: rate(demo_http_requests_total[1m])")
print("=" * 50)
results = prom_query('sum by (host)(rate(demo_http_requests_total[1m]))')
for r in results:
    host = r["metric"].get("host", "unknown")
    value = float(r["value"][1])
    print(f"  {host:<12} {value:.1f} req/sec")

print()

# Query 4: Max CPU over the last 2 minutes
print("📊 Query: max_over_time(demo_cpu_usage[2m])")
print("=" * 50)
results = prom_query('max by (host)(max_over_time(demo_cpu_usage[2m]))')
for r in results:
    host = r["metric"].get("host", "unknown")
    value = float(r["value"][1])
    print(f"  {host:<12} peak CPU={value:.1f}%")

print()
print("💡 Notice how PromQL lets you aggregate, filter, and compute over time windows.")
print("   This is much more natural than SQL for time-series data!")

## ⚡ Caching Query Results with Redis

In production, dashboard queries hit the time-series DB repeatedly for the same data.  
Adding a **Redis cache** in front of the query path avoids redundant work.

The key insight: queries separated by 10 seconds cover **almost identical data** — the only difference is the newest 10 seconds. We can cache the historical part and only query the fresh data.

```
Dashboard → Query Service → Redis Cache (hit?) → Prometheus (miss only)
```

In [ ]:
# Demonstrate caching Prometheus query results in Redis

r = get_redis_client()

def cached_prom_query(query: str, ttl_seconds: int = 15) -> tuple[list, bool]:
    """
    Query Prometheus with a Redis cache layer.
    Identical queries within the TTL window return cached results.
    """
    import hashlib
    cache_key = f"prom_cache:{hashlib.md5(query.encode()).hexdigest()}"

    # Check cache first
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # cache HIT

    # Cache miss — query Prometheus
    results = prom_query(query)

    # Store in Redis with TTL
    r.setex(cache_key, ttl_seconds, json.dumps(results, default=str))

    return results, False  # cache MISS

DEMO_Q = "avg by (region)(demo_cpu_usage)"

# Start from a clean slate so re-running this cell always shows MISS, HIT, HIT.
import hashlib as _h
r.delete(f"prom_cache:{_h.md5(DEMO_Q.encode()).hexdigest()}")

# A 300s TTL rather than the 15s default: the point being demonstrated is that
# an identical query inside the TTL window is served from Redis, and a 15s window
# turns that into a race against however long three HTTP round trips take on a
# busy machine. Widening the window does not weaken the claim — it removes the
# only thing that could make the claim intermittently untrue for the wrong reason.
timings, hits = [], []
for n in (1, 2, 3):
    start = time.time()
    results, hit = cached_prom_query(DEMO_Q, ttl_seconds=300)
    elapsed = (time.time() - start) * 1000
    timings.append(elapsed)
    hits.append(hit)
    print(f"Query {n}: {'🟢 HIT' if hit else '🔴 MISS'} — {elapsed:.1f}ms")

t1, t2, t3 = timings
if t2 < t1:
    print(f"\n📊 The cached read was {t1 / t2:.1f}× faster than the Prometheus round trip.")
else:
    print(f"\n📊 Both were sub-millisecond here ({t1:.1f}ms vs {t2:.1f}ms) — on a loaded")
    print("   Prometheus the gap is orders of magnitude, not a rounding error.")

# The invariant worth asserting is the cache behaviour, not the wall-clock timing
# (timings are noisy on a laptop; correctness is not).
assert hits == [False, True, True], f"expected MISS, HIT, HIT — got {hits}"
print()
print("💡 In production, a busy dashboard with 10 panels refreshing every 5s")
print("   would generate 120 queries/minute. Caching reduces DB load dramatically.")
print()
print("👀 Open RedisInsight (http://localhost:5540) to see the cached keys!")


## 📉 Rollups: Serving Long-Range Queries Fast

Querying raw 10-second data for a 30-day dashboard is brutal:
- 30 days × 86,400 sec/day ÷ 10 = **259,200 data points per series**
- × 1,000 servers = **259 million rows** for one panel!

The solution: **pre-compute aggregates** at coarser resolutions.

| Resolution | Retention | Points per series over its own retention | Points it can serve for a 30-day chart |
|-----------|-----------|------------------------------------------|-----------------------------------------|
| Raw (10s) | 2 days | 17,280 | — raw data is already gone |
| 1-minute | 2 weeks | 20,160 | — only 14 of the 30 days exist |
| 1-hour | 90 days | 2,160 | 720 |
| 1-day | 2 years | 730 | 30 |

A 30-day chart therefore *has* to come from the **hourly** tier: 720 points.
Had you kept raw 10-second data for the full 30 days it would be
30 × 86,400 ÷ 10 = **259,200 points per series** — **360× more** to scan, for a chart
that is only ~1,000 pixels wide. Every point beyond the pixel count is wasted work.

> Note the resolutions and the retentions are two independent choices. The middle column is
> "how many points this tier holds in total"; the right column is "how many of those a 30-day
> query can actually use". Confusing the two is how downsampling tables end up self-contradictory.


In [ ]:
# Demonstrate rollups: compute 1-minute aggregates from raw data

conn = get_db_connection()
cursor = conn.cursor()

# Create a rollup table for 1-minute aggregates
cursor.execute("""
    CREATE TABLE IF NOT EXISTS metrics_1m_rollup (
        metric_name TEXT NOT NULL,
        host TEXT NOT NULL,
        bucket TIMESTAMPTZ NOT NULL,
        avg_value DOUBLE PRECISION,
        min_value DOUBLE PRECISION,
        max_value DOUBLE PRECISION,
        sample_count INTEGER,
        PRIMARY KEY (metric_name, host, bucket)
    )
""")
conn.commit()

# Compute the rollup from raw data
cursor.execute("""
    INSERT INTO metrics_1m_rollup (metric_name, host, bucket, avg_value, min_value, max_value, sample_count)
    SELECT
        metric_name,
        labels->>'host' AS host,
        date_trunc('minute', ts) AS bucket,
        AVG(value),
        MIN(value),
        MAX(value),
        COUNT(*)
    FROM metrics_raw
    WHERE metric_name = 'cpu_usage'
    GROUP BY metric_name, labels->>'host', date_trunc('minute', ts)
    ON CONFLICT (metric_name, host, bucket) DO UPDATE SET
        avg_value = EXCLUDED.avg_value,
        min_value = EXCLUDED.min_value,
        max_value = EXCLUDED.max_value,
        sample_count = EXCLUDED.sample_count
""")
conn.commit()

# Compare sizes
cursor.execute("SELECT COUNT(*) FROM metrics_raw WHERE metric_name = 'cpu_usage'")
raw_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM metrics_1m_rollup")
rollup_count = cursor.fetchone()[0]

print(f"📊 Raw data points:    {raw_count:,}")
print(f"📊 1-minute rollups:   {rollup_count:,}")
if rollup_count > 0:
    print(f"📊 Compression ratio:  {raw_count / rollup_count:.0f}×")

print()

# Show sample rollup data
cursor.execute("""
    SELECT host, bucket, round(avg_value::numeric, 1) as avg_cpu,
           round(min_value::numeric, 1) as min_cpu,
           round(max_value::numeric, 1) as max_cpu,
           sample_count
    FROM metrics_1m_rollup
    ORDER BY bucket DESC, host
    LIMIT 10
""")
print(f"{'Host':<12} {'Bucket':<22} {'Avg':>6} {'Min':>6} {'Max':>6} {'N':>4}")
print("-" * 65)
for row in cursor.fetchall():
    print(f"{row[0]:<12} {str(row[1]):<22} {row[2]:>6} {row[3]:>6} {row[4]:>6} {row[5]:>4}")

print()
print("💡 Rollups are LOSSY — you can't recover individual 10s values from the avg.")
print("   But for dashboards showing 30-day trends, you don't need that detail!")

conn.close()

## 📚 Summary

### Key Takeaways

1. **Metrics** have four parts: name, labels, value, timestamp. A **series** is one unique name+labels combo over time.
2. **Metric types** matter: gauges fluctuate, counters only go up (use `rate()`), histograms show distributions.
3. **Prometheus pulls** metrics from HTTP endpoints — your app exposes `/metrics` and Prometheus scrapes it.
4. **Postgres can't handle** 5M writes/sec — time-series databases use append-only writes and time-based partitioning.
5. **Redis caching** speeds up dashboard queries by avoiding redundant DB hits.
6. **Rollups** pre-compute aggregates so 30-day queries scan 720 points instead of 259,200.
7. **Percentiles never average.** Merge histogram buckets across instances first, *then* call
   `histogram_quantile()` — and remember the answer is an interpolation whose precision is
   capped by your bucket boundaries.

### Next Up

In **Notebook 2**, we'll build **alerting rules and thresholds** — how to define conditions that trigger notifications when your system is in trouble.
